# 🪖 Gurkha Pension Enquiry System  
### Intelligent Rule-Based Conversational Agent

**Module:** AI / Intelligent Systems  
**Type:** Coursework 2  
**Approach:** Rule-Based NLP with Explainability  

---

This notebook implements a conversational AI agent designed to assist Gurkha pensioners with enquiries such as:

- Pension payments  
- Required documents  
- ECHS membership  
- Life certificate renewal  
- Contact information  

The system uses:
- Rule-based intent detection  
- Weighted keyword matching  
- Phrase detection  
- Context memory  
- Confidence scoring  
- Explainability  

---

In [23]:
import re
import random
import datetime

## 🧠 Knowledge Base

The knowledge base stores:
- intents
- keywords (with weights)
- phrases (high priority)
- responses

This represents the agent’s internal world model.

In [24]:
KNOWLEDGE_BASE = {

    # ── GREETING ──────────────────────────────────────────────────────────────
    "greeting": {
        "phrases": [
            "good morning", "good afternoon", "good evening",
            "how are you", "how r u", "what's up", "hey there"
        ],
        "keywords": {
            "hello": 3, "hi": 3, "hey": 3, "greetings": 3,
            "morning": 2, "afternoon": 2, "evening": 2,
            "howdy": 2, "namaste": 3, "salute": 2
        },
        "weight_threshold": 3,
        "responses": [
            (
                "Hello! Welcome to the Gurkha Pension Enquiry Service. "
                "I'm here to help you with pension queries, documentation, payments, "
                "and more. How may I assist you today?"
            ),
            (
                "Good day! You've reached the Gurkha Pension Enquiry helpdesk. "
                "Please feel free to ask me about your pension status, required documents, "
                "ECHS membership, or anything else. What can I help you with?"
            ),
            (
                "Hello and welcome! I'm your Gurkha Pension virtual assistant. "
                "I can guide you through pension payments, documentation, life certificates, "
                "and office enquiries. How can I be of service today?"
            ),
        ],
    },

    # ── FAREWELL ──────────────────────────────────────────────────────────────
    "farewell": {
        "phrases": [
            "good bye", "see you", "see you later", "take care",
            "that's all", "that is all", "no more questions",
            "nothing else", "i am done", "i'm done"
        ],
        "keywords": {
            "bye": 3, "goodbye": 3, "farewell": 3, "exit": 3,
            "quit": 3, "thanks": 2, "thank": 2, "thankyou": 3,
            "cheers": 2, "done": 1
        },
        "weight_threshold": 3,
        "responses": [
            (
                "Thank you for contacting the Gurkha Pension Enquiry Service. "
                "We hope your query was resolved. Have a wonderful day — Jai Gurkha!"
            ),
            (
                "It was a pleasure assisting you today. If you need further help, "
                "don't hesitate to reach out. Take care — Jai Gurkha!"
            ),
            (
                "Thank you for getting in touch. Wishing you all the best. "
                "Feel free to return if you have more questions. Jai Gurkha!"
            ),
        ],
    },

    # ── CONTACT INFORMATION ───────────────────────────────────────────────────
    "contact_info": {
        "phrases": [
            "contact number", "phone number", "telephone number",
            "email address", "office address", "how to contact",
            "get in touch", "reach you", "reach the office",
            "contact details", "contact information"
        ],
        "keywords": {
            "contact": 3, "phone": 3, "telephone": 3, "call": 2,
            "email": 3, "address": 2, "location": 2, "office": 2,
            "reach": 2, "fax": 2, "helpline": 3, "number": 1
        },
        "weight_threshold": 3,
        "responses": [
            (
                "Here are the contact details for the Gurkha Pension Office:\n\n"
                "  📞  Phone     : +44 (0)1264 381 515\n"
                "  📧  Email     : gurkha.pension@mod.gov.uk\n"
                "  🏢  Address   : Gurkha Pension Office, Trenchard Lines,\n"
                "                  Upavon, Pewsey, Wiltshire, SN9 6BE, UK\n"
                "  📠  Fax       : +44 (0)1264 381 516\n\n"
                "Please quote your Pension Reference Number when getting in touch. "
                "Is there anything else I can help you with?"
            ),
        ],
    },

    # ── OFFICE TIMINGS ────────────────────────────────────────────────────────
    "office_timings": {
        "phrases": [
            "office hours", "opening hours", "office timing",
            "when are you open", "what time do you open",
            "working hours", "office open", "office close"
        ],
        "keywords": {
            "timing": 3, "timings": 3, "hours": 2, "open": 2,
            "close": 2, "schedule": 2, "when": 1, "time": 1,
            "available": 2, "working": 2
        },
        "weight_threshold": 4,
        "responses": [
            (
                "The Gurkha Pension Office operates during the following hours:\n\n"
                "  🕘  Monday – Thursday : 09:00 – 17:00 \n"
                "  🕘  Friday            : 09:00 – 16:30 \n"
                "  ❌  Saturday & Sunday  : Closed\n"
                "For urgent matters outside office hours, please send an email to "
                "gurkha.pension@mod.gov.uk and a case officer will respond on the next "
                "working day. Is there anything else I can assist you with?"
            ),
        ],
    },

    # ── PENSION PAYMENT STATUS ────────────────────────────────────────────────
    "pension_payment": {
        "phrases": [
            "pension payment", "pension not received", "pension not arrived",
            "payment not received", "did not receive pension",
            "payment missing", "missing payment", "pension delayed",
            "when is my pension", "pension date", "payment date",
            "check my pension", "pension status", "track pension"
        ],
        "keywords": {
            "payment": 3, "pension": 2, "received": 2, "receive": 2,
            "missing": 3, "delayed": 3, "late": 2, "status": 2,
            "when": 1, "date": 1, "track": 2, "check": 2,
            "arrived": 2, "credited": 2, "transferred": 2
        },
        "weight_threshold": 5,
        "responses": [
            (
                "I understand your concern regarding your pension payment. "
                "Here's what you should know:\n\n"
                "  📅  Gurkha pensions are typically paid on the last working day of each month.\n"
                "  🏦  Payments are made directly into your registered bank account.\n"
                "  ⏳  International transfers may take 2–5 additional working days.\n\n"
                "If your payment is overdue by more than 5 working days, please:\n"
                "  1. Check your bank account and any pending notifications.\n"
                "  2. Confirm your bank details are up to date with the office.\n"
                "  3. Contact us at +44 (0)1264 381 515 or gurkha.pension@mod.gov.uk "
                "     quoting your Pension Reference Number.\n\n"
                "Would you like guidance on updating your bank details or anything else?"
            ),
        ],
    },

    # ── DOCUMENTS (triggers branching flow — see DOCUMENT FLOW below) ─────────
    "documents_general": {
        "phrases": [
            "what documents", "which documents", "documents required",
            "documents needed", "paperwork needed", "forms required",
            "what do i need", "what papers", "submission documents",
            "documents to submit", "list of documents"
        ],
        "keywords": {
            "document": 3, "documents": 3, "paperwork": 3,
            "papers": 2, "form": 2, "forms": 2, "certificate": 2,
            "required": 2, "needed": 2, "need": 1, "submit": 2,
            "application": 2, "list": 1
        },
        "weight_threshold": 4,
        # No direct response — triggers a follow-up question (see document flow)
        "responses": [],
    },

    # ── DOCUMENTS: NEW PENSION ────────────────────────────────────────────────
    "documents_new_pension": {
        "phrases": [
            "new pension", "apply for pension", "pension application",
            "start pension", "fresh pension", "first time pension",
            "begin pension", "new applicant", "pension for the first time"
        ],
        "keywords": {
            "new": 3, "fresh": 3, "first": 2, "apply": 2,
            "application": 2, "start": 2, "begin": 2, "register": 2,
            "enroll": 2, "enrol": 2
        },
        "weight_threshold": 5,
        "responses": [
            (
                "For a **New Pension Application**, you will need to submit the "
                "following documents to the Gurkha Pension Office:\n\n"
                "  📋  Required Documents:\n"
                "  1.  Completed AF B7814 Pension Application Form (signed)\n"
                "  2.  Copy of Discharge Certificate (Red Book / AF B108)\n"
                "  3.  Valid Passport (photo page) or National Identity Card\n"
                "  4.  Proof of Bank Account (bank letter or statement — last 3 months)\n"
                "  5.  Proof of Address (utility bill, rental agreement — last 3 months)\n"
                "  6.  Two recent passport-sized photographs\n"
                "  7.  NI Number (if based in the UK) or equivalent tax ID\n"
                "  8.  Marriage Certificate (if claiming spousal benefits)\n\n"
                "  📬  Submit to: gurkha.pension@mod.gov.uk or by post to the office address.\n\n"
                "Would you like information on processing times or anything else?"
            ),
        ],
    },

    # ── DOCUMENTS: FAMILY / WIDOW PENSION ─────────────────────────────────────
    "documents_family_pension": {
        "phrases": [
            "family pension", "widow pension", "widows pension",
            "pension for wife", "pension for family", "spouse pension",
            "dependent pension", "pension after death", "pension for widow",
            "pension for spouse", "bereaved pension"
        ],
        "keywords": {
            "family": 3, "widow": 3, "widows": 3, "wife": 3,
            "spouse": 3, "dependent": 3, "bereaved": 3,
            "death": 2, "died": 2, "deceased": 2, "passed": 2,
            "survivor": 2, "children": 2, "orphan": 2
        },
        "weight_threshold": 5,
        "responses": [
            (
                "For a **Family / Widow Pension Application**, please prepare the "
                "following documents:\n\n"
                "  📋  Required Documents:\n"
                "  1.  Death Certificate of the serving/retired Gurkha (official copy)\n"
                "  2.  Marriage Certificate (original or certified copy)\n"
                "  3.  Widow's / Dependent's Passport or National ID\n"
                "  4.  Proof of Bank Account in the claimant's name\n"
                "  5.  Completed AF B7814 Pension Claim Form (family section)\n"
                "  6.  Children's Birth Certificates (if claiming for dependants under 18)\n"
                "  7.  Proof of Address (utility bill or government-issued letter)\n"
                "  8.  Two passport-sized photographs of the claimant\n\n"
                "  ⚠️  Note: Claims should be submitted within 3 months of bereavement "
                "to avoid delays in payment.\n\n"
                "I'm truly sorry for your loss. Please don't hesitate to contact the "
                "office directly if you need further support — they are here to help."
            ),
        ],
    },

    # ── ECHS MEMBERSHIP ───────────────────────────────────────────────────────
    "echs_membership": {
        "phrases": [
            "echs membership", "echs card", "ex servicemen health",
            "health scheme", "medical card", "echs enrolment",
            "echs registration", "echs benefits", "echs renewal",
            "how to get echs", "apply for echs"
        ],
        "keywords": {
            "echs": 5, "health": 2, "medical": 2, "card": 2,
            "membership": 2, "scheme": 2, "enrol": 2, "enroll": 2,
            "register": 2, "benefits": 2, "hospital": 2, "clinic": 2
        },
        "weight_threshold": 4,
        "responses": [
            (
                "The **Ex-Servicemen Contributory Health Scheme (ECHS)** provides "
                "medical coverage for retired Gurkha personnel and their dependants.\n\n"
                "  🩺  ECHS Membership Details:\n"
                "  • Eligibility  : Retired Gurkha soldiers with a valid pension\n"
                "  • Coverage     : Outpatient, inpatient, specialist referrals\n"
                "  • Card Type    : Smart Card issued by Ministry of Defence\n\n"
                "  📋  Documents Required for ECHS Enrolment:\n"
                "  1.  Pension Payment Order (PPO)\n"
                "  2.  Discharge Certificate (Red Book)\n"
                "  3.  Passport-sized photographs (4 copies)\n"
                "  4.  Valid Passport or ID\n"
                "  5.  Proof of Address\n"
                "  6.  Completed ECHS Enrolment Form (Form 2)\n\n"
                "  📞  To apply or renew, contact the nearest ECHS Polyclinic "
                "or call the pension office at +44 (0)1264 381 515.\n\n"
                "Is there anything else I can help you with regarding ECHS?"
            ),
        ],
    },

    # ── LIFE CERTIFICATE ──────────────────────────────────────────────────────
    "life_certificate": {
        "phrases": [
            "life certificate", "life cert", "annual certificate",
            "proof of life", "liveness certificate", "yearly certificate",
            "existence certificate", "renew life certificate",
            "submit life certificate", "life certificate renewal"
        ],
        "keywords": {
            "life": 3, "certificate": 3, "liveness": 3, "annual": 2,
            "yearly": 2, "renew": 2, "renewal": 2, "proof": 2,
            "alive": 2, "existence": 2, "submission": 2, "submit": 2
        },
        "weight_threshold": 5,
        "responses": [
            (
                "The **Life Certificate** (also called a Proof of Life or Liveness "
                "Certificate) must be submitted annually to continue receiving your "
                "Gurkha pension without interruption.\n\n"
                "  📅  Submission Deadline : By 31st October each year\n"
                "  📌  Who Must Submit     : All pension recipients (including family/widow)\n\n"
                "  ✅  Accepted Methods:\n"
                "  1.  Visit the nearest British Gurkha Camp or Embassy for a witnessed signature.\n"
                "  2.  Have the certificate signed by a Notary Public or Magistrate.\n"
                "  3.  Visit a UK Post Office that offers the 'Know Your Customer' service.\n\n"
                "  📋  Documents Needed:\n"
                "  • Completed Life Certificate Form (issued by the pension office)\n"
                "  • Valid Passport or National ID (for identity verification)\n\n"
                "  ⚠️  Warning: Failure to submit by the deadline may result in "
                "temporary suspension of pension payments.\n\n"
                "Would you like help with anything else, such as downloading the form "
                "or finding your nearest witness location?"
            ),
        ],
    },

    # ── PENSION AMOUNT / CALCULATION ──────────────────────────────────────────
    "pension_amount": {
        "phrases": [
            "how much pension", "pension amount", "pension rate",
            "how is pension calculated", "pension calculation",
            "pension entitlement", "how much will i get",
            "pension increase", "pension rise", "annual pension"
        ],
        "keywords": {
            "amount": 3, "rate": 3, "calculation": 3, "calculate": 3,
            "entitlement": 3, "much": 2, "get": 1, "receive": 1,
            "increase": 2, "rise": 2, "annual": 2, "how": 1
        },
        "weight_threshold": 5,
        "responses": [
            (
                "Gurkha pension amounts are calculated based on several factors:\n\n"
                "  🔢  Key Factors:\n"
                "  • Length of service (years served in the British Army)\n"
                "  • Rank at discharge\n"
                "  • Pension scheme (pre-2007 Gurkha Pension Scheme or post-2007 AFPS)\n"
                "  • Additional allowances (e.g., disability, overseas supplement)\n\n"
                "  📈  Annual Review:\n"
                "  Pension rates are reviewed each April in line with UK inflation "
                "(Consumer Price Index — CPI).\n\n"
                "  ℹ️  For a personalised pension calculation or to understand your "
                "specific entitlement, please contact the Gurkha Pension Office directly:\n"
                "  📞  +44 (0)1264 381 515\n"
                "  📧  gurkha.pension@mod.gov.uk\n\n"
                "Is there anything else I can help you with?"
            ),
        ],
    },

    # ── CHANGE OF DETAILS ──────────────────────────────────────────────────────
    "change_details": {
        "phrases": [
            "change address", "update address", "change bank details",
            "update bank", "change account", "new bank account",
            "change name", "update details", "change details",
            "update information", "bank account change"
        ],
        "keywords": {
            "change": 3, "update": 3, "new": 2, "amend": 3,
            "address": 2, "bank": 2, "account": 2, "name": 2,
            "details": 2, "information": 2, "modify": 3, "edit": 2
        },
        "weight_threshold": 5,
        "responses": [
            (
                "To update your personal or banking details with the Gurkha Pension "
                "Office, please follow these steps:\n\n"
                "  📋  Process:\n"
                "  1.  Write a formal letter or email to the office requesting the change.\n"
                "  2.  Clearly state your full name, Pension Reference Number, and "
                "the details to be changed.\n\n"
                "  📂  Supporting Documents Required:\n"
                "  • Bank Account Change : New bank statement or official bank letter "
                "(showing name, account number, sort code / IBAN)\n"
                "  • Address Change      : Utility bill or official letter (last 3 months)\n"
                "  • Name Change         : Deed poll or marriage/divorce certificate\n\n"
                "  📧  Send to : gurkha.pension@mod.gov.uk\n"
                "  📠  Fax to  : +44 (0)1264 381 516\n\n"
                "  ⏳  Changes typically take 5–10 working days to process.\n\n"
                "Is there anything else I can assist you with?"
            ),
        ],
    },

    # ── FALLBACK ───────────────────────────────────────────────────────────────
    "fallback": {
        "phrases": [],
        "keywords": {},
        "weight_threshold": 0,
        "responses": [
            (
                "I'm sorry, I didn't quite catch that. Could you please rephrase your "
                "question? For example, you could ask about:\n"
                "  • Pension payments or status\n"
                "  • Required documents\n"
                "  • Life certificate renewal\n"
                "  • ECHS health scheme\n"
                "  • Office contact details or timings"
            ),
            (
                "I want to make sure I give you the right information. Could you "
                "provide a little more detail about your query? I'm here to help with "
                "pension payments, documentation, ECHS, and more."
            ),
            (
                "That's not something I have a specific answer for right now. "
                "For complex or personalised queries, I'd recommend contacting our "
                "team directly at +44 (0)1264 381 515 or gurkha.pension@mod.gov.uk. "
                "Is there something else I can help you with?"
            ),
        ],
    },
}



## 🔍 Intent Detection Engine

This module performs:

1. Text normalisation  
2. Phrase matching (high priority)  
3. Weighted keyword scoring  
4. Threshold filtering  
5. Priority-based selection  
6. Confidence calculation  

This ensures accurate and explainable reasoning.

In [25]:
class IntentDetector:
    """
    Rule-based intent detection using phrase matching and weighted keywords.
    Returns the detected intent, confidence score, and matched evidence.
    """

    # Priority order for intents — more specific intents rank higher.
    # When two intents have identical scores, the one earlier in this list wins.
    PRIORITY_ORDER = [
        "documents_new_pension",
        "documents_family_pension",
        "documents_general",
        "life_certificate",
        "echs_membership",
        "pension_payment",
        "pension_amount",
        "change_details",
        "contact_info",
        "office_timings",
        "farewell",
        "greeting",
        "fallback",
    ]

    def __init__(self, knowledge_base: dict):
        self.kb = knowledge_base

    # Common English contractions → expanded form
    # Expanded before punctuation removal so "haven't" → "have not", not "haven t"
    CONTRACTIONS = {
        "haven't": "have not", "hasn't": "has not", "didn't": "did not",
        "doesn't": "does not", "don't": "do not", "isn't": "is not",
        "wasn't": "was not", "weren't": "were not", "can't": "cannot",
        "couldn't": "could not", "won't": "will not", "wouldn't": "would not",
        "shouldn't": "should not", "i'm": "i am", "i've": "i have",
        "i'll": "i will", "i'd": "i would", "it's": "it is",
        "that's": "that is", "there's": "there is", "they're": "they are",
        "you're": "you are", "we're": "we are", "aren't": "are not",
    }

    def normalise(self, text: str) -> str:
        """Lowercase, expand contractions, strip punctuation for consistent matching."""
        text = text.lower()
        # Expand contractions before removing punctuation
        for contraction, expansion in self.CONTRACTIONS.items():
            text = text.replace(contraction, expansion)
        text = re.sub(r"[^\w\s]", " ", text)  # Remove remaining punctuation
        text = re.sub(r"\s+", " ", text).strip()  # Collapse whitespace
        return text

    def detect(self, user_input: str) -> dict:
        """
        Main detection method.
        Returns a result dict with: intent, confidence, matched_phrases,
        matched_keywords, and all_scores.
        """
        normalised = self.normalise(user_input)
        scores = {}
        evidence = {}

        for intent_name, intent_data in self.kb.items():
            if intent_name == "fallback":
                continue

            matched_phrases = []
            matched_keywords = {}
            score = 0

            # ── Step 1: Phrase Matching (high-value signal) ──────────────────
            # Phrases are worth 6 points each — they are very specific signals.
            for phrase in intent_data.get("phrases", []):
                if phrase in normalised:
                    matched_phrases.append(phrase)
                    score += 6

            # ── Step 2: Keyword Matching (weighted) ──────────────────────────
            # Split input into individual words and match against keyword dict.
            words = normalised.split()
            for word in words:
                weight = intent_data["keywords"].get(word, 0)
                if weight > 0:
                    matched_keywords[word] = weight
                    score += weight

            # ── Step 3: Apply Threshold Gate ─────────────────────────────────
            # Only consider intents that pass their minimum threshold.
            threshold = intent_data.get("weight_threshold", 3)
            if score >= threshold:
                scores[intent_name] = score
                evidence[intent_name] = {
                    "matched_phrases": matched_phrases,
                    "matched_keywords": matched_keywords,
                    "raw_score": score,
                }

        # ── Step 4: Priority-Aware Selection ─────────────────────────────────
        if not scores:
            return {
                "intent": "fallback",
                "confidence": 0.0,
                "matched_phrases": [],
                "matched_keywords": {},
                "all_scores": {},
            }

        # Find the highest raw score
        max_score = max(scores.values())

        # Among all intents with the max score, pick the highest-priority one
        top_intents = [i for i, s in scores.items() if s == max_score]
        selected_intent = min(
            top_intents,
            key=lambda i: self.PRIORITY_ORDER.index(i)
            if i in self.PRIORITY_ORDER else 999,
        )

        # ── Step 5: Confidence Normalisation ─────────────────────────────────
        # Confidence = raw score / (raw score + 10) maps to (0, 1)
        # This gives a natural saturation curve — avoids overconfidence.
        raw = scores[selected_intent]
        confidence = raw / (raw + 10)

        return {
            "intent": selected_intent,
            "confidence": round(confidence, 3),
            "matched_phrases": evidence[selected_intent]["matched_phrases"],
            "matched_keywords": evidence[selected_intent]["matched_keywords"],
            "all_scores": scores,
        }


## 🧩 Context Memory

The chatbot maintains context to support multi-step interactions.

Example:
User: "What documents do I need?"  
Bot: "Which type?"  
User: "1"  

This requires memory of the previous state.

In [26]:
class ContextMemory:
    """
    Manages the agent's short-term conversational memory.
    Tracks:
      - pending_intent  : An intent awaiting user follow-up
      - last_intent     : The most recent resolved intent
      - last_result     : The full detection result for explainability
      - turn_count      : Number of dialogue turns
      - session_start   : Timestamp of session start
    """

    def __init__(self):
        self.pending_intent: str | None = None   # Waiting for user's choice
        self.last_intent: str | None = None
        self.last_result: dict | None = None
        self.turn_count: int = 0
        self.session_start: str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def set_pending(self, intent: str):
        self.pending_intent = intent

    def clear_pending(self):
        self.pending_intent = None

    def record(self, result: dict):
        self.last_intent = result["intent"]
        self.last_result = result
        self.turn_count += 1

    def is_pending(self) -> bool:
        return self.pending_intent is not None

    def explain_last(self) -> str:
        """Generate a human-readable explanation of the last decision."""
        if not self.last_result:
            return "No previous response to explain."

        r = self.last_result
        intent = r["intent"].replace("_", " ").title()
        confidence_pct = int(r["confidence"] * 100)
        phrases = r.get("matched_phrases", [])
        keywords = r.get("matched_keywords", {})
        all_scores = r.get("all_scores", {})

        lines = [
            "━" * 56,
            "  🔍  EXPLAINABILITY REPORT — Last Response",
            "━" * 56,
            f"  Detected Intent   : {intent}",
            f"  Confidence Score  : {confidence_pct}%",
        ]

        if phrases:
            lines.append(f"  Matched Phrases   : {', '.join(phrases)}")
        else:
            lines.append("  Matched Phrases   : (none — keyword match only)")

        if keywords:
            kw_display = ", ".join(f'"{k}" (+{v})' for k, v in keywords.items())
            lines.append(f"  Matched Keywords  : {kw_display}")
        else:
            lines.append("  Matched Keywords  : (none)")

        if all_scores:
            lines.append("  All Intent Scores :")
            for intent_name, score in sorted(all_scores.items(), key=lambda x: -x[1]):
                label = intent_name.replace("_", " ").title()
                lines.append(f"    • {label:<30} → {score}")

        lines.append("━" * 56)
        return "\n".join(lines)


## 💬 Response Engine

This module:

- Selects responses based on intent  
- Handles branching logic (documents flow)  
- Manages follow-up questions  
- Supports explainability  

It ensures conversational behaviour.

In [27]:
class ResponseEngine:
    """
    Generates the agent's reply given a detection result and context memory.
    Manages the multi-step document flow and fallback handling.
    """

    # Document follow-up question shown after "documents_general" is detected
    DOCUMENT_FOLLOW_UP = (
        "Of course! To give you the correct document list, could you please "
        "tell me which type of pension you're enquiring about?\n\n"
        "  1️⃣   New Pension Application\n"
        "  2️⃣   Family / Widow Pension\n\n"
        "Please reply with 1, 2, or describe your situation."
    )

    def __init__(self, knowledge_base: dict, context: ContextMemory):
        self.kb = knowledge_base
        self.context = context

    def get_response(self, detection_result: dict, raw_input: str) -> str:
        """
        Core response selection method.
        Returns the final response string to be shown to the user.
        """
        intent = detection_result["intent"]
        normalised_input = raw_input.lower().strip()

        # ── Handle Explainability Request ────────────────────────────────────
        if self._is_explain_request(normalised_input):
            return self.context.explain_last()

        # ── Handle Pending Document Flow ─────────────────────────────────────
        if self.context.is_pending() and self.context.pending_intent == "documents_general":
            return self._resolve_document_choice(normalised_input, detection_result)

        # ── Handle Document General (triggers follow-up) ──────────────────
        if intent == "documents_general":
            self.context.set_pending("documents_general")
            return self.DOCUMENT_FOLLOW_UP

        # ── Handle Document Specific (direct detection, no follow-up needed) ─
        if intent in ("documents_new_pension", "documents_family_pension"):
            self.context.clear_pending()
            return self._pick_response(intent)

        # ── Standard Intent Response ─────────────────────────────────────────
        return self._pick_response(intent)

    def _resolve_document_choice(self, user_input: str, detection: dict) -> str:
        """
        Resolve the user's follow-up choice in the document branching flow.
        Accepts: "1", "new", "new pension", "2", "family", "widow", etc.
        """
        self.context.clear_pending()

        # Check if detection already resolved to a specific document intent
        if detection["intent"] == "documents_new_pension":
            return self._pick_response("documents_new_pension")
        if detection["intent"] == "documents_family_pension":
            return self._pick_response("documents_family_pension")

        # Numeric / keyword fallback parsing
        if re.search(r"\b1\b|new|fresh|first|apply|application", user_input):
            return self._pick_response("documents_new_pension")
        if re.search(r"\b2\b|family|widow|wife|spouse|bereaved|death|died", user_input):
            return self._pick_response("documents_family_pension")

        # Could not resolve — ask again
        self.context.set_pending("documents_general")
        return (
            "I'm sorry, I didn't quite understand your selection. "
            "Please reply with:\n\n"
            "  1️⃣   for New Pension Application\n"
            "  2️⃣   for Family / Widow Pension"
        )

    def _pick_response(self, intent: str) -> str:
        """Randomly select a response from the intent's response list."""
        responses = self.kb.get(intent, {}).get("responses", [])
        if not responses:
            return self._pick_response("fallback")
        return random.choice(responses)

    @staticmethod
    def _is_explain_request(text: str) -> bool:
        """Detect if the user is asking the agent to explain itself."""
        explain_patterns = [
            r"\bwhy\b", r"\bexplain\b", r"\bhow did you\b",
            r"\bwhy did you\b", r"\bjustify\b", r"\breasoning\b",
            r"\bhow do you know\b", r"\bwhat made you\b",
        ]
        return any(re.search(pat, text) for pat in explain_patterns)


## 🤖 Intelligent Agent Design

The chatbot follows the **Sense–Think–Respond model**:

- Sense → user input  
- Think → intent detection + reasoning  
- Respond → generate output  

This aligns with rational agent design principles.

In [28]:
class GurkhaEnquiryAgent:
    """
    Top-level intelligent agent for Gurkha Pension Enquiries.

    Design Principle: Sense → Think → Respond (STR loop)
      Sense  : Read user input
      Think  : Detect intent, score confidence, check context
      Respond: Generate appropriate professional reply
    """

    AGENT_NAME = "Gurkha Pension Assistant"
    VERSION = "1.0.0"

    def __init__(self):
        self.kb = KNOWLEDGE_BASE
        self.context = ContextMemory()
        self.detector = IntentDetector(self.kb)
        self.responder = ResponseEngine(self.kb, self.context)

    # ── Public Interface ─────────────────────────────────────────────────────

    def respond(self, user_input: str) -> str:
        if not user_input.strip():
            return "I notice you didn't type anything. How can I help you today?"
    
        # ✅ STEP 0: handle explain FIRST (DO NOT TOUCH MEMORY)
        if self.responder._is_explain_request(user_input.lower()):
            return self.context.explain_last()
    
        # SENSE → THINK
        detection = self.detector.detect(user_input)
    
        # RESPOND
        response = self.responder.get_response(detection, user_input)
    
        # ONLY NOW store memory (after response is decided)
        if detection["confidence"] > 0.2:
            self.context.record(detection)
    
        return response
        
    def is_exit_intent(self, user_input: str) -> bool:
        """Check if the user wants to end the session."""
        exit_words = {"exit", "quit", "bye", "goodbye", "farewell"}
        normalised = user_input.lower().strip()
        return normalised in exit_words or any(w in normalised for w in exit_words)

    def welcome_message(self) -> str:
        """Return the opening banner displayed at session start."""
        border = "═" * 60
        return (
            f"\n{border}\n"
            f"  🪖  {self.AGENT_NAME}  v{self.VERSION}\n"
            f"  British Army Gurkha Pension Enquiry — Virtual Helpdesk\n"
            f"{border}\n"
            f"  Type your question below and press ENTER to continue.\n"
            f"  Type 'why' or 'explain' to see how I reached my answer.\n"
            f"  Type 'exit' or 'bye' to end the session.\n"
            f"{border}\n"
        )

    def run(self):
        """
        Main interactive dialogue loop.
        Runs until the user types an exit command.
        """
        print(self.welcome_message())

        while True:
            try:
                user_input = input("  You   > ").strip()
            except (KeyboardInterrupt, EOFError):
                print("\n\n  [Session interrupted. Goodbye — Jai Gurkha!]\n")
                break

            if not user_input:
                continue

            # Check for exit before full processing
            if self.is_exit_intent(user_input):
                farewell = self.respond(user_input)
                print(f"\n  Agent > {farewell}\n")
                break

            # Full STR cycle
            reply = self.respond(user_input)

            # Pretty-print the agent reply (handles multi-line responses)
            print()
            for line in reply.split("\n"):
                print(f"  Agent > {line}" if line else "")
            print()



## ▶️ Run the Chatbot

Use the cell below to interact with the chatbot.

Type:
- normal questions → get answers  
- "why" or "explain" → see reasoning  
- "exit" → end session  

In [29]:
agent = GurkhaEnquiryAgent()
agent.run()


════════════════════════════════════════════════════════════
  🪖  Gurkha Pension Assistant  v1.0.0
  British Army Gurkha Pension Enquiry — Virtual Helpdesk
════════════════════════════════════════════════════════════
  Type your question below and press ENTER to continue.
  Type 'why' or 'explain' to see how I reached my answer.
  Type 'exit' or 'bye' to end the session.
════════════════════════════════════════════════════════════



  You   >  hi



  Agent > Hello! Welcome to the Gurkha Pension Enquiry Service. I'm here to help you with pension queries, documentation, payments, and more. How may I assist you today?



  You   >  why



  Agent > ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Agent >   🔍  EXPLAINABILITY REPORT — Last Response
  Agent > ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Agent >   Detected Intent   : Greeting
  Agent >   Confidence Score  : 23%
  Agent >   Matched Phrases   : (none — keyword match only)
  Agent >   Matched Keywords  : "hi" (+3)
  Agent >   All Intent Scores :
  Agent >     • Greeting                       → 3
  Agent > ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



  You   >  exit



  Agent > It was a pleasure assisting you today. If you need further help, don't hesitate to reach out. Take care — Jai Gurkha!



In [31]:
# =========================
# AUTOMATED TEST CONVERSATIONS
# =========================

def run_test_conversation(agent, conversation, title):
    print("\n" + "="*60)
    print(f"🧪 {title}")
    print("="*60)

    for turn in conversation:
        print(f"\nYou   > {turn}")
        response = agent.respond(turn)

        for line in response.split("\n"):
            print(f"Agent > {line}" if line else "")


# Initialize agent
agent = GurkhaEnquiryAgent()


# =========================
# CONVERSATION 1 — SUCCESS (DOCUMENT FLOW + EXPLAINABILITY)
# =========================
conv1 = [
    "I am a new pension what documents I need",
    "why",
    "1",
    "why"
]


# =========================
# CONVERSATION 2 — SUCCESS (PENSION PAYMENT + EXPLAINABILITY)
# =========================
conv2 = [
    "My pension not received",
    "why",
    "is it delayed because of bank issues?",
    "why"
]


# =========================
# CONVERSATION 3 — SUCCESS (ECHS + EXPLAINABILITY)
# =========================
conv3 = [
    "What is echs membership",
    "why",
    "how do I apply for it",
    "why"
]


# =========================
# CONVERSATION 4 — FAILURE (VAGUE INPUT + EXPLANATION ATTEMPT)
# =========================
conv4 = [
    "money problem",
    "why",
    "I mean pension money issue",
    "why"
]


# =========================
# CONVERSATION 5 — OUT OF DOMAIN + EXPLAINABILITY
# =========================
conv5 = [
    "what is visa status",
    "why",
    "can you help with pension instead?",
    "why"
]

# =========================
# RUN ALL TESTS
# =========================
run_test_conversation(agent, conv1, "Conversation 1 — Document Flow (Success)")
run_test_conversation(agent, conv2, "Conversation 2 — Payment Issue (Success)")
run_test_conversation(agent, conv3, "Conversation 3 — Explainability (Success)")
run_test_conversation(agent, conv4, "Conversation 4 — Vague Input (Failure)")
run_test_conversation(agent, conv5, "Conversation 5 — Out-of-Domain (Failure)")


🧪 Conversation 1 — Document Flow (Success)

You   > I am a new pension what documents I need
Agent > Of course! To give you the correct document list, could you please tell me which type of pension you're enquiring about?

Agent >   1️⃣   New Pension Application
Agent >   2️⃣   Family / Widow Pension

Agent > Please reply with 1, 2, or describe your situation.

You   > why
Agent > ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Agent >   🔍  EXPLAINABILITY REPORT — Last Response
Agent > ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Agent >   Detected Intent   : Documents General
Agent >   Confidence Score  : 50%
Agent >   Matched Phrases   : what documents
Agent >   Matched Keywords  : "documents" (+3), "need" (+1)
Agent >   All Intent Scores :
Agent >     • Documents General              → 10
Agent >     • Documents New Pension          → 9
Agent > ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

You   > 1
Agent > For a **New Pension Application**, you wil